# PPO 训练吞吐探针（TPU / GPU / CPU 三设备）

**用途**：量出 PPO 一个梯度步的真实耗时（s/step），回答两个问题：

1. **GPU 配额用完后切 Kaggle TPU（独立 20h/周）值不值？**
2. **已实装的两项优化到底值多少？** —— ref 前向缓存、`.item()` 批量化。

**判据线**（来自真实训练数据，148 步/轮 = 37 chunks × 4 epochs）：

| 基准 | s/step |
|---|---|
| 本机 CPU 实测 | ~4.70（700 s/轮） |
| Kaggle GPU 实测 | ~0.47（70 s/轮） |

**先读这一段再跑**：

- 本 notebook **不需要克隆仓库**，探针是自包含的；下面的 `%%writefile` 单元会把脚本原样写盘。
- **TPU 运行时千万不要 `pip install --upgrade torch`** —— Kaggle TPU 镜像里的 torch 与
  torch_xla 是严格配对的一组，装错任一个 TPU 直接不可用。下面的环境单元会替你检查。
- 想拿 GPU 对照数，就用 GPU 运行时跑同一个 notebook（把 `--device tpu` 换成 `--device cuda`）。


In [ ]:
# ── 环境探测：本 runtime 是 GPU 还是 TPU？配对的 torch / torch_xla 是否可用？ ──
import os
import sys

import torch

print(f"python  : {sys.version.split()[0]}")
print(f"torch   : {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

# Kaggle TPU VM 自带配好对的 torch_xla；PJRT_DEVICE 指定后端。
os.environ.setdefault("PJRT_DEVICE", "TPU")
_tpu_kind = None
try:
    import torch_xla
    import torch_xla.core.xla_model as xm

    _xla_dev = xm.xla_device()
    _tpu_kind = str(_xla_dev)
    print(f"torch_xla: {getattr(torch_xla, '__version__', '?')}")
    print(f"xla dev  : {_xla_dev}")
except Exception as e:
    print(f"torch_xla: 不可用（{type(e).__name__}: {e}）")

print()
if _tpu_kind:
    print("=> 本 runtime 有 TPU。直接跑下面的 TPU 单元。")
elif torch.cuda.is_available():
    print("=> 本 runtime 是 GPU（无 TPU）。要测 GPU 对照数，把下面单元的 --device tpu 换成 --device cuda。")
else:
    print("=> 本 runtime 既无 TPU 也无 GPU，只能跑 CPU 对照。")
    print("   请在 Kaggle 新建 notebook 时把 Accelerator 选成 'TPU VM v3-8'。")


## 写入探针脚本（原样复制，逐字节一致）


In [ ]:
%%writefile tpu_probe.py
"""PPO 吞吐探针 —— 自包含单文件，不依赖仓库任何模块。

两个用途：

  ① **TPU 可行性**：在 Kaggle TPU notebook 上量 PPO 单步耗时 (s/step)，回答
     「GPU 配额用完后切 TPU 能不能净增 20h/周算力」。判据线来自真实训练数据：
        本机 CPU   ~700 s/轮  -> 4.7 s/step
        Kaggle GPU ~70  s/轮  -> 0.47 s/step
     （148 steps = 37 chunks x 4 epochs，见 tmp/<course>/training_log.jsonl）

  ② **性能优化效果**：分离测出各优化项的收益（2026-09-10）
       A 纯 fwd+bwd+step（固定 shape）        基准
       B = A + 每步 8 次 .item() 同步         同步税
       C = B + 变化尾块                        XLA 重编译税（TPU 独有）
       D1 = B + ref 每步重算（旧 engine 行为） 双前向税
       D2 = B + ref 预计算缓存（新 engine 行为）ref 成本降到 1/epochs
     D1 vs D2 就是 ppo/engine.py 本次落地的 ref_cache 优化的实测收益
     （数值逐位不变，见 tests/test_ppo_kickstart_cache.py）。

用法：
    Kaggle TPU：  !python tpu-probe.py --device tpu
    本机对照：     python tpu-probe.py --device cpu --quick
    只测优化效果：  python tpu-probe.py --device cpu --skip-shapes

模型与 models/student.py 逐位一致（BN-free ConvMixer-Lite h=64 d=8，stem 权重已折进
1/255）；训练循环与 ppo/engine.py::ppo_update 的算子结构等价。
"""

from __future__ import annotations

import argparse
import os
import time
from typing import Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---------------------------------------------------------------- 常量（= schema.py）
BOARD = 26
OBS_CHANNELS = 14
SCALAR_DIM = 19
MOVE_DIM = 5
FIRE_DIM = 2
H = 64
D = 8
HEAD_HIDDEN = 128

MB = 512  # 与 rl-config / 课程一致
CHUNKS = 37  # c5-margin 实测每轮 36-37 块
EPOCHS = 4

CPU_LINE = 4.7  # 700 s / 148 steps
GPU_LINE = 0.47  #  70 s / 148 steps


# ---------------------------------------------------------------- 模型（内联副本）
class ConvMixerBlock(nn.Module):
    def __init__(self, h: int):
        super().__init__()
        self.dw = nn.Conv2d(h, h, 5, padding=2, groups=h, bias=True)
        self.pw = nn.Conv2d(h, h, 1, bias=True)

    def forward(self, x):
        return x + F.relu(self.pw(F.relu(self.dw(x))))


def coord_channels(board: int, device) -> torch.Tensor:
    r = torch.arange(board, dtype=torch.float32, device=device) / (board - 1)
    x = r.repeat(board, 1)
    y = x.t()
    return (torch.stack([x, y]) * 255).round().to(torch.uint8)


class PPOStudent(nn.Module):
    """= models/student.py::PPOStudent（BN-free，stem 含 1/255 折入）。"""

    def __init__(
        self,
        in_ch=OBS_CHANNELS,
        board=BOARD,
        scalar_dim=SCALAR_DIM,
        h=H,
        d=D,
        head_hidden=HEAD_HIDDEN,
    ):
        super().__init__()
        self.board, self.head_hidden = board, head_hidden
        self.stem = nn.Conv2d(in_ch + 2, h, 3, padding=1, bias=True)
        self.blocks = nn.ModuleList([ConvMixerBlock(h) for _ in range(d)])
        self.fc = nn.Linear(h + scalar_dim, head_hidden, bias=True)
        self.move_head = nn.Linear(head_hidden, MOVE_DIM, bias=True)
        self.fire_head = nn.Linear(head_hidden, FIRE_DIM, bias=True)
        self.value_head = nn.Linear(head_hidden, 1, bias=True)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_uniform_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity="relu")
                nn.init.zeros_(m.bias)
        with torch.no_grad():
            self.stem.weight.mul_(1.0 / 255.0)

    def features(self, obs, scalars):
        coords = coord_channels(self.board, obs.device).float().unsqueeze(0)
        x = torch.cat([obs.float(), coords.expand(obs.shape[0], -1, -1, -1)], dim=1)
        x = F.relu(self.stem(x))
        for b in self.blocks:
            x = b(x)
        x = x.mean(dim=(2, 3))
        return F.relu(self.fc(torch.cat([x, scalars], dim=1)))

    def forward(self, obs, scalars):
        h = self.features(obs, scalars)
        return self.move_head(h), self.fire_head(h), self.value_head(h)


# ---------------------------------------------------------------- batch 合成
def make_chunk(b: int, rng: np.random.Generator) -> dict:
    return {
        "obs": rng.integers(0, 256, (b, OBS_CHANNELS, BOARD, BOARD), dtype=np.uint8),
        "scalars": rng.standard_normal((b, SCALAR_DIM)).astype(np.float32),
        "a_move": rng.integers(0, MOVE_DIM, (b,), dtype=np.int64),
        "a_fire": rng.integers(0, FIRE_DIM, (b,), dtype=np.int64),
        "lp_move": rng.standard_normal(b).astype(np.float32),
        "lp_fire": rng.standard_normal(b).astype(np.float32),
        "adv": rng.standard_normal(b).astype(np.float32),
        "ret": rng.standard_normal(b).astype(np.float32),
        "mask": np.ones((b, MOVE_DIM + FIRE_DIM), dtype=np.float32),
    }


# ---------------------------------------------------------------- 算子（= ppo/common.py）
def masked_logsoftmax(logits, mask):
    big = torch.tensor(1e9, device=logits.device, dtype=logits.dtype)
    m = mask.to(logits.dtype)
    return F.log_softmax(logits + (1.0 - m) * (-big), dim=-1)


def cat_logprob(a, lp):
    return lp.gather(1, a.unsqueeze(1)).squeeze(1)


def cat_entropy(lp):
    return -(lp.exp() * lp).sum(dim=-1).mean()


# ---------------------------------------------------------------- 设备适配
class Backend:
    """统一 CPU / CUDA / TPU 的 step 与统计同步语义。"""

    def __init__(self, kind: str):
        self.kind = kind
        self.xm: Any = None
        if kind == "tpu":
            os.environ.setdefault("PJRT_DEVICE", "TPU")
            import torch_xla.core.xla_model as xm

            self.xm = xm
            self.device = xm.xla_device()
        elif kind == "cuda":
            self.device = torch.device("cuda")
            torch.backends.cuda.matmul.allow_tf32 = True
            torch.backends.cudnn.allow_tf32 = True
        else:
            self.device = torch.device("cpu")

    def to(self, m):
        return m.to(self.device)

    def step(self, opt):
        if self.kind == "tpu":
            # 官方口径：梯度跨副本归约 + 显式图执行边界。裸 opt.step() 在
            # XRT/GSPMD 路径上会漏掉执行边界（静默不更新）。
            self.xm.optimizer_step(opt)
        else:
            opt.step()

    def mark(self):
        if self.kind == "tpu":
            self.xm.mark_step()


# ---------------------------------------------------------------- 训练步
def prepare(bk: Backend, chunks: list[dict]) -> list[dict]:
    """numpy chunks -> 设备张量，只做一次（= engine 的 tensored_chunks）。"""
    out = []
    for c in chunks:
        out.append({k: torch.from_numpy(v).to(bk.device) for k, v in c.items()})
    bk.mark()
    return out


def build_ref_cache(bk: Backend, ref_model, dev_chunks: list[dict]) -> list:
    """新 engine 行为：每 chunk 只算一次 ref，按索引复用（跨 epoch 不变）。"""
    out = []
    with torch.no_grad():
        for e in dev_chunks:
            rm, rf, _ = ref_model(e["obs"], e["scalars"])
            m = e["mask"]
            out.append(
                (
                    masked_logsoftmax(rm, m[:, :MOVE_DIM]),
                    masked_logsoftmax(rf, m[:, MOVE_DIM : MOVE_DIM + FIRE_DIM]),
                )
            )
    bk.mark()
    return out


def run_steps(
    bk: Backend,
    model,
    opt,
    dev_chunks,
    epochs,
    *,
    sync_mode: str = "batched",
    ref_model=None,
    ref_cache=None,
) -> tuple[float, int]:
    """跑 epochs x len(chunks) 个梯度步，返回 (总秒数, 步数)。

    sync_mode: none（不同步，算力下限）/ per-item（旧引擎 8 次 .item()）/
    batched（新引擎 1 次 stack().tolist()）。per-item - batched 即该优化的净值。


    ref_model 非空 -> 旧行为（每步现算 ref）；ref_cache 非空 -> 新行为（预计算复用）。
    """
    n_steps = len(dev_chunks) * epochs
    t0 = time.time()
    for _ep in range(epochs):
        for ci, e in enumerate(dev_chunks):
            obs, sc, mask = e["obs"], e["scalars"], e["mask"]
            mv, fr, val = model(obs, sc)
            move_logp = masked_logsoftmax(mv, mask[:, :MOVE_DIM])
            fire_logp = masked_logsoftmax(fr, mask[:, MOVE_DIM : MOVE_DIM + FIRE_DIM])
            lp_new = cat_logprob(e["a_move"], move_logp) + cat_logprob(e["a_fire"], fire_logp)
            lp_old = e["lp_move"] + e["lp_fire"]
            ratio = torch.exp(lp_new - lp_old)
            adv = e["adv"]
            surr1 = ratio * adv
            surr2 = torch.clamp(ratio, 0.8, 1.2) * adv
            policy_loss = -torch.min(surr1, surr2).mean()
            value_loss = F.mse_loss(val.squeeze(-1), e["ret"])
            entropy = cat_entropy(move_logp) + cat_entropy(fire_logp)
            loss = policy_loss + 1.0 * value_loss - 0.01 * entropy

            kick_mean = torch.zeros((), device=bk.device)
            ref_pair = ref_cache[ci] if ref_cache is not None else None
            if ref_pair is None and ref_model is not None:
                with torch.no_grad():
                    rm, rf, _ = ref_model(obs, sc)
                    ref_pair = (
                        masked_logsoftmax(rm, mask[:, :MOVE_DIM]),
                        masked_logsoftmax(rf, mask[:, MOVE_DIM : MOVE_DIM + FIRE_DIM]),
                    )
            if ref_pair is not None:
                ref_move, ref_fire = ref_pair
                kl_m = (move_logp.exp() * (move_logp - ref_move)).sum(dim=-1)
                kl_f = (fire_logp.exp() * (fire_logp - ref_fire)).sum(dim=-1)
                kick_mean = (kl_m + kl_f).mean()
                loss = loss + 1.0 * kick_mean

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            bk.step(opt)

            if sync_mode == "per-item":
                # 旧 ppo/engine.py 的真实形态：每步 8 次主机同步
                _ = float(policy_loss.item())
                _ = float(value_loss.item())
                _ = float(entropy.item())
                _ = float(ratio.mean().item())
                _ = float(adv.mean().item())
                _ = float(e["ret"].mean().item())
                _ = float(lp_new.mean().item())
                _ = float(val.mean().item())
            elif sync_mode == "batched":
                # 新形态（ppo/common.sync_scalars）：8 个标量一次 stack().tolist()
                _ = torch.stack(
                    [
                        policy_loss.detach().reshape(()),
                        value_loss.detach().reshape(()),
                        entropy.detach().reshape(()),
                        ratio.mean().detach().reshape(()),
                        adv.mean().detach().reshape(()),
                        e["ret"].mean().detach().reshape(()),
                        lp_new.mean().detach().reshape(()),
                        val.mean().detach().reshape(()),
                    ]
                ).tolist()
            else:
                bk.mark()
    bk.mark()
    return time.time() - t0, n_steps


def bench_case(
    bk,
    chunks,
    epochs,
    *,
    sync_mode="batched",
    ref_cache=None,
    ref_model=None,
    warmup=1,
    repeat=2,
    seed=0,
):
    """warmup 掉 XLA 编译，再取 repeat 次最小值。"""
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = bk.to(PPOStudent())
    opt = torch.optim.Adam(model.parameters(), lr=1.5e-4)
    dev_chunks = prepare(bk, chunks)
    for _ in range(warmup):
        run_steps(
            bk,
            model,
            opt,
            dev_chunks[: min(2, len(dev_chunks))],
            1,
            sync_mode=sync_mode,
            ref_model=ref_model,
            ref_cache=ref_cache,
        )
    best = float("inf")
    n = 0
    for _ in range(repeat):
        s, n = run_steps(
            bk,
            model,
            opt,
            dev_chunks,
            epochs,
            sync_mode=sync_mode,
            ref_model=ref_model,
            ref_cache=ref_cache,
        )
        best = min(best, s)
    return best / n, n


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--device", default="tpu", choices=["tpu", "cuda", "cpu"])
    ap.add_argument("--chunks", type=int, default=CHUNKS)
    ap.add_argument("--epochs", type=int, default=EPOCHS)
    ap.add_argument("--quick", action="store_true", help="本地自检：小规模（4 块 x 1 epoch）")
    ap.add_argument(
        "--skip-shapes", action="store_true", help="跳过 C 段（变化 shape，TPU 重编译税）"
    )
    ap.add_argument("--skip-ref", action="store_true", help="跳过 D1/D2 段（ref 双前向对照）")
    args = ap.parse_args()

    if args.quick:
        args.chunks, args.epochs = 4, 1

    bk = Backend(args.device)
    n_par = sum(p.numel() for p in PPOStudent().parameters())
    n_steps_iter = 148  # 37 x 4，用于把 s/step 折成 s/轮
    print("=" * 72)
    print(f"PPOStudent params={n_par:,}  device={args.device} -> {bk.device}")
    print(f"chunks={args.chunks} x epochs={args.epochs} = {args.chunks * args.epochs} steps")
    print("=" * 72)

    rng = np.random.default_rng(20260910)
    fixed = [make_chunk(MB, rng) for _ in range(args.chunks)]
    # 变化尾块：模拟真实每轮不同的 n mod 512（c5-margin 74 轮出现 66 种）
    var_tails = [230, 39, 456, 365, 424, 65]

    cases: list[tuple[str, float]] = []

    s, n = bench_case(bk, fixed, args.epochs, sync_mode="none")
    cases.append(("A 纯 fwd+bwd+step（固定 shape, 无同步）", s))
    print(f"  A: {s * 1000:7.0f} ms/step")

    s, n = bench_case(bk, fixed, args.epochs, sync_mode="per-item")
    cases.append(("B_old = A + 8x .item()/step（旧引擎）", s))
    print(f"  B_old: {s * 1000:7.0f} ms/step")

    s, n = bench_case(bk, fixed, args.epochs, sync_mode="batched")
    cases.append(("B_new = A + 1x stack().tolist()（新引擎）", s))
    print(f"  B_new: {s * 1000:7.0f} ms/step")

    if not args.skip_shapes:
        worst = 0.0
        for tail in var_tails:
            ch = [make_chunk(MB, rng) for _ in range(max(1, args.chunks - 1))] + [
                make_chunk(tail, rng)
            ]
            s, n = bench_case(bk, ch, args.epochs, sync_mode="batched", repeat=1)
            worst = max(worst, s)
            print(f"  C tail={tail:4d}: {s * 1000:7.0f} ms/step")
        cases.append(("C = B + 变化尾块（XLA 重编译最坏值）", worst))

    d1 = d2 = None
    if not args.skip_ref:
        torch.manual_seed(0)
        ref = bk.to(PPOStudent()).eval()
        for p in ref.parameters():
            p.requires_grad_(False)
        dev_fixed = prepare(bk, fixed)
        cache = build_ref_cache(bk, ref, dev_fixed)

        d1, n = bench_case(bk, fixed, args.epochs, sync_mode="batched", ref_model=ref)
        cases.append(("D1 = B + ref 每步重算（旧 engine）", d1))
        print(f"  D1: {d1 * 1000:7.0f} ms/step  (ref 每步现算)")

        d2, n = bench_case(bk, fixed, args.epochs, sync_mode="batched", ref_cache=cache)
        cases.append(("D2 = B + ref 预计算缓存（新 engine）", d2))
        print(f"  D2: {d2 * 1000:7.0f} ms/step  (ref 每 chunk 一次)")

    print("=" * 72)
    print(f"{'case':<44}{'s/step':>10}{'s/轮':>10}")
    print("-" * 72)
    for name, s in cases:
        print(f"{name:<44}{s:>10.3f}{s * n_steps_iter:>10.0f}")
    print("-" * 72)
    print(f"{'本机 CPU 实测线（用户口径）':<44}{CPU_LINE:>10.2f}{700:>10.0f}")
    print(f"{'Kaggle GPU 实测线（用户口径）':<44}{GPU_LINE:>10.2f}{70:>10.0f}")
    print("=" * 72)

    got = dict(cases)
    b_old = got["B_old = A + 8x .item()/step（旧引擎）"]
    b = got["B_new = A + 1x stack().tolist()（新引擎）"]
    print()
    ga = got["A 纯 fwd+bwd+step（固定 shape, 无同步）"]
    print(f"[优化效果]  同步税 旧 B_old-A = {(b_old - ga) * 1000:+.0f} ms/step")
    print(f"[优化效果]  同步税 新 B_new-A = {(b - ga) * 1000:+.0f} ms/step")
    print(
        f"[优化效果]  .item()->批量化   = {(b_old - b) * 1000:+.0f} ms/step "
        f"({(b_old - b) / b_old * 100:.1f}% of B_old)"
    )
    if d1 and d2:
        print(
            f"[优化效果]  ref 缓存 D1-D2 = {(d1 - d2) * 1000:+.0f} ms/step "
            f"({(d1 - d2) / d1 * 100:.1f}% of D1)"
        )
        print(
            f"[优化效果]  真实形态端到端 = {d1:.3f} -> {d2:.3f} s/step "
            f"({d1 / d2:.2f}x)，折算 {(d1 - d2) * n_steps_iter:.0f}s/轮"
        )
    print()
    if b < GPU_LINE:
        print(
            f"[TPU 判据] B={b:.3f}s/step < GPU 线 {GPU_LINE} —— TPU 可**顶替** GPU，值得全量接入。"
        )
    elif b < CPU_LINE:
        print(
            f"[TPU 判据] B={b:.3f}s/step < CPU 线 {CPU_LINE} —— TPU 可**叠加**在 GPU 配额之后"
            f"（20h/周净增）。值得接入。"
        )
    else:
        print(f"[TPU 判据] B={b:.3f}s/step >= CPU 线 {CPU_LINE} —— TPU 不比本机 CPU 快。")
        print("           若 C 段远高于 B，先修尾块 shape（pad 到 mb，消掉每轮重编译）再测。")


if __name__ == "__main__":
    main()



In [ ]:
# 校验写盘的探针可运行（不实跑，只 import + 打帮助）
import subprocess

r = subprocess.run([sys.executable, "tpu_probe.py", "--help"], capture_output=True, text=True)
print("exit =", r.returncode)
print(r.stdout[:400] or r.stderr[:400])


## 开始测量


In [ ]:
# ── 主测：TPU ──
# chunks/epochs 取小值即可：耗时是**每步**量,与总步数无关;epochs=4 保住 ref 缓存的
# 摊销比(=1/epochs,与真实 37 chunks x 4 epochs 同比例)。
# 若 C 段(变化尾块,每块触发一次 XLA 编译)太慢,加 --skip-shapes 先看 A/B/D 段。
!python tpu_probe.py --device tpu --chunks 6 --epochs 4


In [ ]:
# ── 可选对照：GPU（需在 GPU runtime 下跑;这是判定"D2 同步税到底值多少"的关键数据）
!python tpu_probe.py --device cuda --skip-shapes


In [ ]:
# ── 可选对照：CPU（同机型的相对量;用于确认"探针在这台机器上的 CPU 数"和你的口径是否吻合）
!python tpu_probe.py --device cpu --quick --skip-shapes


## 怎么读结果

探针输出一张表 + 几行 `[优化效果]` / `[TPU 判据]`。按这个顺序读：

### 1. 先看 `[优化效果] .item()->批量化`（B_old − B_new）

**这是本次最关键的未测项。** CPU 上实测只有 +28 ms/step（0.8%，噪声内），因为 CPU 的
`.item()` 数据已在主机内存、几乎免费。**CUDA / TPU 上每次 `.item()` 都是全设备同步**
（强制 drain 未执行的 kernel 队列），量级完全不同。

- 显著为正（几十~几百 ms/step）⇒ 「GPU 利用率仅 2.9%」的锅基本坐实，优化已直接见效。
- 仍 ≈ 0 ⇒ 瓶颈另有其处，转向内存布局（channels_last）或批量（mb），**不要动精度**。

### 2. 再看 `[优化效果] ref 缓存 D1-D2`

CPU 实测 −861 / −953 ms/step（19.8% / 21.8%，1.25× / 1.28×），两次独立复现。
GPU 上如果远小于这个比例，说明 ref 前向已被良好 overlap 隐藏 —— 那 D1 在 GPU 上的收益要重新评估。

### 3. 最后看 `[TPU 判据] B=...`

- `B < 0.47 s/step` ⇒ TPU 可**顶替** GPU，值得全量接入。
- `0.47 < B < 4.70` ⇒ TPU 可**叠加**在 GPU 配额之后（每周净增 20h）。
- `B > 4.70` ⇒ 比本机 CPU 还慢，接入无收益；先看 C 段是否被 XLA 重编译税占满，
  那样的话先修尾块 shape（pad 到固定 mb）再复测。

### 4. C 段（变化尾块）只在 TPU 上有意义

`chunk_episodes` 全局 shuffle 后按 mb=512 切片，尾块 = `n mod 512`；实测 c5-margin 74 轮出现
**66 个不同尾块尺寸、0 轮整除**。XLA 按 shape 缓存图 ⇒ 每轮一个新 shape = 一次新编译。
C 段与 B 段的差值就是这笔税的实测值。GPU/CPU 上这个差应当接近 0。

---

**注意**：本 notebook 只做**测量**，不改任何仓库代码。结果请贴回
`plan/ppo-optimization.plan.md` §4/§6 的判读段。
